# CrewAI Orchestration, Flows & Memory Mechanics

---
## 0. Install Dependencies

In [ ]:
%%capture
!pip install 'crewai[tools]>=0.80.0' crewai-tools langchain-openai pydantic pyyaml

---
## 1. API Keys & Global Config

In [ ]:
import os
from google.colab import userdata

# ── Set via Colab Secrets (left sidebar → 🔑) OR paste directly ──
try:
    os.environ["OPENAI_API_KEY"]  = userdata.get("OPENAI_API_KEY")
    os.environ["SERPER_API_KEY"]  = userdata.get("SERPER_API_KEY")
except Exception:
    os.environ["OPENAI_API_KEY"]  = "sk-..."   # ← paste your key here
    os.environ["SERPER_API_KEY"]  = "..."       # ← serper.dev key here

# Global model choice — cheapest available
MODEL = "gpt-4o-mini"
TOPIC = "The impact of Large Language Models on software engineering"

print("✅ Keys loaded")
print(f"📌 Model : {MODEL}")
print(f"📌 Topic : {TOPIC}")

---
## 2. Load YAML Configurations



In [ ]:
import yaml
from pathlib import Path

# ── Fallback YAML content (used if files not uploaded) ────────────────────
_AGENTS_YAML_FALLBACK = """
researcher:
  role: "Senior Research Analyst"
  goal: >
    Discover accurate, comprehensive, and well-sourced information on the
    assigned topic. Always cite sources and flag uncertain claims clearly.
  backstory: >
    You are a meticulous research analyst with 15 years of experience.
    You prefer breadth over depth and always include multiple perspectives.
  verbose: true
  allow_delegation: false

fact_checker:
  role: "Senior Fact-Check Analyst"
  goal: >
    Verify every factual claim. Return a structured verdict:
    VERIFIED, UNVERIFIED, or CONTRADICTED.
  backstory: >
    Former investigative journalist. Skeptical by nature, binary verdicts.
  verbose: true
  allow_delegation: false

writer:
  role: "Content Writer & Editor"
  goal: >
    Transform verified research into a polished, publication-ready article.
  backstory: >
    Award-winning science writer. Never publishes CONTRADICTED claims.
  verbose: true
  allow_delegation: false
"""

_TASKS_YAML_FALLBACK = """
research_task:
  description: >
    Research the following topic thoroughly: {topic}
    Use SerperDevTool for web search and attempt deep-dive research with
    the unreliable_research_tool. If it times out, fall back to SerperDevTool.
    Return a JSON object with: topic, key_findings (list), sources (list),
    confidence_level (float), tool_failure_occurred (bool).
  expected_output: >
    A concise 3-bullet summary followed by a JSON block with the required fields.
  agent: researcher

fact_check_task:
  description: >
    Fact-check the researcher output. For each claim use fact_check_tool.
    Return JSON: claims_checked, verdicts (list), overall_verdict (PASS/FAIL),
    recommendation (PROCEED_TO_WRITING or SEND_BACK_TO_RESEARCH).
  expected_output: >
    JSON with overall_verdict and recommendation. Machine-readable only.
  agent: fact_checker

write_article_task:
  description: >
    Write a publication-ready article on: {topic}
    Structure: Headline, Lead, Body (3-4 paragraphs), Conclusion.
    Skip CONTRADICTED claims. Tag UNVERIFIED with [UNVERIFIED].
  expected_output: >
    Complete markdown article, 400-600 words, with metadata footer.
  agent: writer
"""

# ── Load from file if present, else use fallback ──────────────────────────
def load_config(filename: str, fallback: str) -> dict:
    path = Path(filename)
    if path.exists():
        print(f"✅ Loaded {filename} from uploaded file")
        return yaml.safe_load(path.read_text())
    else:
        print(f"⚠️  {filename} not found — using built-in fallback")
        return yaml.safe_load(fallback)

agents_config = load_config("agents.yaml", _AGENTS_YAML_FALLBACK)
tasks_config  = load_config("tasks.yaml",  _TASKS_YAML_FALLBACK)

print(f"\nAgents defined : {list(agents_config.keys())}")
print(f"Tasks  defined : {list(tasks_config.keys())}")

---
## 3. Custom Tools

### 3.1 `UnreliableResearchTool` — fails 50% of the time
Simulates a flaky external API (rate limits, slow DB, etc.).

### 3.2 `FactCheckTool` — returns structured verdicts
Used by Phase 3 Flow for deterministic routing.

In [ ]:
import random, time
from crewai.tools import BaseTool
from pydantic import BaseModel, Field
from typing import Type


# ── Tool 1: Unreliable Research Tool ─────────────────────────────────────
class UnreliableResearchInput(BaseModel):
    query: str = Field(..., description="The research query to investigate")


class UnreliableResearchTool(BaseTool):
    """
    Raises TimeoutError 50% of the time to simulate a flaky data source.

    Phase 1 goal: observe how Sequential vs Hierarchical crews behave
    when this tool fails mid-execution.
    """
    name: str = "unreliable_research_tool"
    description: str = (
        "Performs deep research on a topic using an external source. "
        "WARNING: may time out 50% of the time."
    )
    args_schema: Type[BaseModel] = UnreliableResearchInput

    def _run(self, query: str) -> str:
        if random.random() < 0.5:
            raise TimeoutError(
                f"[UnreliableResearchTool] Timed out for query: '{query}'. "
                "Use fallback tool instead."
            )
        time.sleep(0.3)  # simulate latency on success
        return (
            f"[UnreliableResearchTool] SUCCESS for '{query}'. "
            "Key findings: Advanced research data retrieved successfully."
        )


# ── Tool 2: Fact Check Tool ───────────────────────────────────────────────
class FactCheckInput(BaseModel):
    claim: str = Field(..., description="The claim to verify")
    source: str = Field(default="web", description="Source context")


class FactCheckTool(BaseTool):
    """
    Returns a structured VERIFIED / UNVERIFIED / CONTRADICTED verdict.
    The Phase 3 Flow router uses this output for deterministic branching.
    """
    name: str = "fact_check_tool"
    description: str = (
        "Validates a factual claim. Returns JSON with verdict: "
        "VERIFIED, UNVERIFIED, or CONTRADICTED."
    )
    args_schema: Type[BaseModel] = FactCheckInput

    def _run(self, claim: str, source: str = "web") -> str:
        word_count = len(claim.split())
        verdict = "VERIFIED" if word_count > 20 else (
                  "UNVERIFIED" if word_count > 10 else "CONTRADICTED")
        return (
            f'{{"verdict": "{verdict}", '
            f'"claim": "{claim[:60]}...", '
            f'"confidence": 0.85, "source": "{source}"}}'
        )


print("✅ UnreliableResearchTool defined (50% TimeoutError)")
print("✅ FactCheckTool defined (VERIFIED / UNVERIFIED / CONTRADICTED)")

# Quick smoke-test
tool = UnreliableResearchTool()
successes = 0
failures = 0
for _ in range(10):
    try:
        tool._run("test query")
        successes += 1
    except TimeoutError:
        failures += 1
print(f"\nSmoke-test (10 runs): {successes} success, {failures} failures")
print("Expected: ~5 each (50% failure rate)")

---
## 4. Agent & Task Factory (shared by Phase 1 & 2)

In [ ]:
from crewai import Agent, Task, Crew, Process
from crewai_tools import SerperDevTool
from langchain_openai import ChatOpenAI


def get_llm(model: str = MODEL) -> ChatOpenAI:
    """Shared LLM factory — always defaults to the cheapest model."""
    return ChatOpenAI(
        model=model,
        temperature=0.1,
        api_key=os.environ["OPENAI_API_KEY"],
    )


def build_agents(cfg: dict, llm) -> dict:
    """
    Instantiate Agents from YAML config.
    Identity/personality lives in YAML; tools are attached in code.
    This separation makes agents portable across projects.
    """
    researcher = Agent(
        role=cfg["researcher"]["role"],
        goal=cfg["researcher"]["goal"],
        backstory=cfg["researcher"]["backstory"],
        tools=[SerperDevTool(), UnreliableResearchTool()],
        llm=llm,
        verbose=True,
        allow_delegation=False,
        max_retry_limit=2,   # retry tool failures up to 2 times before escalating
    )
    fact_checker = Agent(
        role=cfg["fact_checker"]["role"],
        goal=cfg["fact_checker"]["goal"],
        backstory=cfg["fact_checker"]["backstory"],
        tools=[FactCheckTool()],
        llm=llm,
        verbose=True,
        allow_delegation=False,
    )
    writer = Agent(
        role=cfg["writer"]["role"],
        goal=cfg["writer"]["goal"],
        backstory=cfg["writer"]["backstory"],
        tools=[],
        llm=llm,
        verbose=True,
        allow_delegation=False,
    )
    return {"researcher": researcher, "fact_checker": fact_checker, "writer": writer}


def build_tasks(tcfg: dict, agents: dict, topic: str) -> list:
    """
    Instantiate Tasks from YAML config.
    Topic is injected via .format() — tasks are reusable templates.
    """
    research_task = Task(
        description=tcfg["research_task"]["description"].format(topic=topic),
        expected_output=tcfg["research_task"]["expected_output"],
        agent=agents["researcher"],
    )
    fact_check_task = Task(
        description=tcfg["fact_check_task"]["description"].format(topic=topic),
        expected_output=tcfg["fact_check_task"]["expected_output"],
        agent=agents["fact_checker"],
        context=[research_task],
    )
    write_task = Task(
        description=tcfg["write_article_task"]["description"].format(topic=topic),
        expected_output=tcfg["write_article_task"]["expected_output"],
        agent=agents["writer"],
        context=[research_task, fact_check_task],
    )
    return [research_task, fact_check_task, write_task]


print("✅ Agent & Task factory functions defined")

---
# Phase 1 — Orchestration & Failure Handling

We run the **same three agents on the same topic** under two process modes and observe how each handles `TimeoutError` from the unreliable tool.

### 1A — Sequential Execution

In [ ]:
print("=" * 60)
print("  PHASE 1A — SEQUENTIAL EXECUTION")
print("=" * 60)
print("""
WHAT TO OBSERVE:
  • Agents run in strict order: Researcher → Fact-Checker → Writer
  • If UnreliableResearchTool raises TimeoutError:
      - CrewAI retries up to max_retry_limit (2) times
      - On exhaustion → Task FAILS → Crew HALTS
      - Fact-Checker and Writer never execute
  • No Manager LLM overhead — cheapest mode
""")

llm_seq = get_llm()
agents_seq = build_agents(agents_config, llm_seq)
tasks_seq  = build_tasks(tasks_config, agents_seq, TOPIC)

crew_sequential = Crew(
    agents=list(agents_seq.values()),
    tasks=tasks_seq,
    process=Process.sequential,
    verbose=True,
    # memory=False here (Phase 2 enables memory)
)

try:
    result_seq = crew_sequential.kickoff(inputs={"topic": TOPIC})
    print("\n" + "="*60)
    print("SEQUENTIAL RESULT:")
    print("="*60)
    print(result_seq)
except TimeoutError as e:
    print(f"\n❌ SEQUENTIAL HALTED — TimeoutError propagated to caller:")
    print(f"   {e}")
    print("   → Fact-Checker and Writer did NOT run")
except Exception as e:
    print(f"\n❌ SEQUENTIAL FAILURE: {type(e).__name__}: {e}")

### 1B — Hierarchical Execution

In [ ]:
print("=" * 60)
print("  PHASE 1B — HIERARCHICAL EXECUTION")
print("=" * 60)
print("""
WHAT TO OBSERVE:
  • A Manager LLM orchestrates all agents dynamically
  • Manager intercepts TimeoutError and can:
      - Instruct Researcher to use SerperDevTool instead
      - Retry with modified approach
      - Mark task best-effort and continue
  • More resilient, but ~40-100% more API calls than Sequential
  • Watch for extra [Manager] log lines between agent turns
""")

llm_hier   = get_llm()
manager_llm = get_llm()   # separate instance for clarity; same cheap model
agents_hier = build_agents(agents_config, llm_hier)
tasks_hier  = build_tasks(tasks_config, agents_hier, TOPIC)

crew_hierarchical = Crew(
    agents=list(agents_hier.values()),
    tasks=tasks_hier,
    process=Process.hierarchical,
    manager_llm=manager_llm,
    verbose=True,
)

try:
    result_hier = crew_hierarchical.kickoff(inputs={"topic": TOPIC})
    print("\n" + "="*60)
    print("HIERARCHICAL RESULT:")
    print("="*60)
    print(result_hier)
except Exception as e:
    print(f"\n❌ HIERARCHICAL FAILURE: {type(e).__name__}: {e}")

### Phase 1 - Questions & Answers

**Q1: What happens when the tool fails in Sequential execution?**

> The `TimeoutError` propagates from the tool → Task executor → Crew runner. CrewAI retries up to `max_retry_limit` (2) times on the same agent. If all retries are exhausted the current Task fails and the **entire Crew halts** — downstream tasks (Fact-Checker, Writer) never run. The error surfaces directly to the `crew.kickoff()` caller with no automatic fallback.

**Q2: How does the Manager LLM handle failure in Hierarchical execution?**

> The Manager LLM receives the failure report as part of the conversation context. It can dynamically decide to: **(1)** retry the Researcher with new instructions ("use only SerperDevTool this time"), **(2)** reassign a subtask to another agent, or **(3)** accept a partial result and continue. This recovery happens without surfacing the error to the caller, making Hierarchical mode significantly more resilient under tool failures.

**Q3: Why is Hierarchical processing significantly more expensive?**

> Every agent delegation requires **one extra Manager API call** (decision) plus **one validation call** (completion check). The Manager also maintains a full conversation thread containing all agent outputs — large context = more tokens. In practice: ~3 API calls in Sequential becomes ~5–7 in Hierarchical for the same 3-task crew. Expect 40–100% more spend.

---
# Phase 2 — YAML Configuration & Memory Behavior

Introduces `memory=True` and deliberately inspects how CrewAI resolves the **backstory vs expected_output** contradiction.

### 2A — The Contradiction (inspect YAML configs)

In [ ]:
print("CONFLICT ANALYSIS")
print("-" * 50)
print("Researcher BACKSTORY says:")
print(f'  "{agents_config["researcher"]["backstory"][:120].strip()}..."')
print()
print("research_task EXPECTED OUTPUT says:")
print(f'  "{tasks_config["research_task"]["expected_output"][:120].strip()}..."')
print()
print("""
RESOLUTION:
  expected_output wins — it is the final instruction in the task's user-turn
  prompt, making it the most proximate directive. The LLM treats backstory
  as "how I tend to work" but treats expected_output as "what I MUST deliver".

INTERNAL PAYLOAD ASSEMBLY:
  System prompt = Role + Goal + Backstory + Tool descriptions
  User turn     = Task description + Context (prev outputs) + Expected output

  Because expected_output is appended LAST in the user turn, it overrides
  any personality-level guidance from the system prompt.
""")

### 2B — Memory-Enabled Crew Execution

In [ ]:
print("=" * 60)
print("  PHASE 2 — MEMORY-ENABLED SEQUENTIAL CREW")
print("=" * 60)
print("""
memory=True enables THREE memory systems:
  1. Short-term  → in-process buffer (RAG over THIS run, resets on kickoff)
  2. Long-term   → SQLite  at ~/.local/share/crewai/<name>/  (persists!)
  3. Entity      → ChromaDB at same path  (named entity vectors, persists!)

Run this cell TWICE to observe memory carry-over:
  Day 1: crew builds initial context
  Day 2: crew retrieves prior summaries from SQLite + ChromaDB
""")

llm_mem = get_llm()
agents_mem = build_agents(agents_config, llm_mem)
tasks_mem  = build_tasks(tasks_config, agents_mem, TOPIC)

crew_memory = Crew(
    agents=list(agents_mem.values()),
    tasks=tasks_mem,
    process=Process.sequential,
    memory=True,                            # ← enable all memory types
    verbose=True,
    embedder={                              # required for vector store
        "provider": "openai",
        "config": {"model": "text-embedding-3-small"},
    },
)

try:
    result_mem = crew_memory.kickoff(inputs={"topic": TOPIC})
    print("\n" + "="*60)
    print("RESULT (memory-enabled):")
    print("="*60)
    print(result_mem)
except Exception as e:
    print(f"❌ {type(e).__name__}: {e}")

### 2C — Inspect Memory Storage

In [ ]:
import sqlite3
from pathlib import Path

# CrewAI stores memory at ~/.local/share/crewai/
memory_root = Path.home() / ".local" / "share" / "crewai"

print("Memory storage root:", memory_root)
print()

if memory_root.exists():
    for p in sorted(memory_root.rglob("*")):
        size = p.stat().st_size if p.is_file() else 0
        print(f"  {'📄' if p.is_file() else '📁'} {p.relative_to(memory_root)}  "
              + (f"({size:,} bytes)" if p.is_file() else ""))
else:
    print("⚠️  Memory directory not found yet — run Phase 2B first")

# Try to read the SQLite long-term memory
db_files = list(memory_root.rglob("*.db")) if memory_root.exists() else []
if db_files:
    print(f"\n📊 Reading long-term memory DB: {db_files[0]}")
    try:
        conn = sqlite3.connect(db_files[0])
        cursor = conn.cursor()
        cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
        tables = cursor.fetchall()
        print(f"Tables: {[t[0] for t in tables]}")
        for table in tables:
            cursor.execute(f"SELECT COUNT(*) FROM {table[0]}")
            count = cursor.fetchone()[0]
            print(f"  {table[0]}: {count} rows")
        conn.close()
    except Exception as e:
        print(f"  Could not read DB: {e}")
else:
    print("\nNo .db files found yet")

### Phase 2 - Questions & Answers

**Q1: Which instruction takes priority — Agent backstory or Task expected_output?**

> **Task `expected_output` always wins.** It is appended as the final instruction in the user-turn prompt, making it the most proximate directive. The backstory shapes reasoning style; `expected_output` defines the contract.

**Q2: How is the final API payload constructed internally?**

> `System prompt = [Role] + [Goal] + [Backstory] + [Tool descriptions]`  
> `User turn = [Task description] + [Context outputs] + [Expected output]`  
> The `expected_output` is always last in the user turn — hence its precedence.

**Q3: What local storage/database is used for memory?**

> **SQLite** (`long_term_memory.db`) for task summaries and outputs.  
> **ChromaDB** (`entity_memory/`) for named entity vector embeddings.  
> Both stored at `~/.local/share/crewai/<crew_name>/`.

**Q4: How does memory persist across multiple runs (Day 1 → Day 2)?**

> Short-term memory (in-process buffer) **resets** each `kickoff()`. Long-term (SQLite) and entity (ChromaDB) memories **persist on disk**. On Day 2, the crew queries SQLite for prior summaries and ChromaDB for related entities — enabling it to build on previous research without redundant API calls.

---
# Phase 3 - Event-Driven Flows & State Management

Converts the static crew into a **CrewAI Flow** with:
- Pydantic `ResearchFlowState` — typed, serializable, debuggable
- `@start / @listen / @router` decorators for event-driven step wiring
- Retry counter guardrail preventing infinite research ↔ fact-check loops

### 3A — Structured State Definition

In [ ]:
from typing import Optional, Literal
from pydantic import BaseModel, Field


class ResearchFlowState(BaseModel):
    """
    WHY STRUCTURED STATE OVER RAW TEXT
    ─────────────────────────────────────
    Raw text:           Structured state:
    ─────────           ────────────────
    No schema           Schema enforced at assignment
    LLM parses output   Router reads typed field directly
    Silent failures     Bad data raises ValidationError
    Hard to debug       JSON-serializable, diffable
    No IDE support      Autocomplete + type checking
    """
    topic: str = Field(default="")

    # ── Step outputs ──────────────────────────────────────────────
    research_output: Optional[str]  = Field(default=None)
    research_json:   Optional[dict] = Field(default=None)
    tool_failure_occurred: bool     = Field(default=False)

    fact_check_output: Optional[str] = Field(default=None)
    fact_check_verdict: Literal["PASS", "FAIL", "PENDING"] = Field(default="PENDING")
    fact_check_recommendation: Literal[
        "PROCEED_TO_WRITING", "SEND_BACK_TO_RESEARCH", "PENDING"
    ] = Field(default="PENDING")

    final_article: Optional[str] = Field(default=None)

    # ── GUARDRAIL FIELDS ──────────────────────────────────────────
    # Without these, FAIL → research → FAIL again = infinite loop
    research_retry_count: int = Field(default=0)
    MAX_RESEARCH_RETRIES: int = Field(default=2)   # hard cap

    # ── Audit trail ───────────────────────────────────────────────
    execution_log: list = Field(default_factory=list)


# Verify the model
s = ResearchFlowState(topic="test")
print("✅ ResearchFlowState schema:")
for name, field in ResearchFlowState.model_fields.items():
    print(f"  {name:<30} {str(field.annotation):<40} default={field.default}")

### 3B — Flow Definition

In [ ]:
import json
from crewai.flow.flow import Flow, listen, start, router


class ResearchDepartmentFlow(Flow[ResearchFlowState]):
    """
    Flow graph:

      @start() → research()
                    │
      @listen(research) → fact_check()
                              │
      @router(fact_check) → route_after_fact_check()
                              │           │           │
                            PASS         FAIL       FAIL+maxRetries
                              │           │               │
                        write_article  research()   escalate_to_human()
                              │        (loop with
                        @listen(...)   retry counter)
    """

    def __init__(self, topic: str):
        super().__init__()
        self.state.topic = topic
        self._llm = get_llm()

    # ─── helpers ────────────────────────────────────────────────────────

    def _run_mini_crew(self, agent: Agent, task: Task) -> str:
        """Run a single-agent crew and return the string result."""
        crew = Crew(
            agents=[agent], tasks=[task],
            process=Process.sequential, verbose=True,
        )
        return str(crew.kickoff())

    def _parse_verdict(self, raw: str) -> tuple[str, str]:
        """Extract verdict and recommendation from raw output."""
        try:
            data = json.loads(raw)
            verdict = data.get("overall_verdict", "FAIL")
            rec     = data.get("recommendation", "SEND_BACK_TO_RESEARCH")
        except json.JSONDecodeError:
            upper = raw.upper()
            verdict = "PASS" if "\"PASS\"" in upper or "OVERALL_VERDICT: PASS" in upper else "FAIL"
            rec     = "PROCEED_TO_WRITING" if verdict == "PASS" else "SEND_BACK_TO_RESEARCH"
        return verdict, rec

    # ─── Step 1: Research ────────────────────────────────────────────────

    @start()
    def research(self):
        """Entry point. On retries, researcher is told what failed."""
        retry_note = (
            f" [RETRY {self.state.research_retry_count}/{self.state.MAX_RESEARCH_RETRIES}]"
            " — previous fact-check FAILED, improve sourcing and depth."
            if self.state.research_retry_count > 0 else ""
        )
        self.state.execution_log.append(
            f"[research] start (retry={self.state.research_retry_count})"
        )

        agent = Agent(
            role=agents_config["researcher"]["role"],
            goal=agents_config["researcher"]["goal"],
            backstory=agents_config["researcher"]["backstory"],
            tools=[SerperDevTool(), UnreliableResearchTool()],
            llm=self._llm, verbose=True, max_retry_limit=2,
        )
        task = Task(
            description=(
                f"Research: {self.state.topic}{retry_note}\n"
                "Return JSON: topic, key_findings (list), sources (list), "
                "confidence_level (float 0-1), tool_failure_occurred (bool)."
            ),
            expected_output="JSON research data. Set tool_failure_occurred=true if timeout.",
            agent=agent,
        )
        result = self._run_mini_crew(agent, task)
        self.state.research_output = result

        # parse structured fields if possible
        try:
            data = json.loads(result)
            self.state.research_json = data
            self.state.tool_failure_occurred = data.get("tool_failure_occurred", False)
        except json.JSONDecodeError:
            self.state.tool_failure_occurred = "tool_failure_occurred" in result.lower()

        self.state.execution_log.append(
            f"[research] done | tool_failure={self.state.tool_failure_occurred}"
        )

    # ─── Step 2: Fact Check ──────────────────────────────────────────────

    @listen(research)
    def fact_check(self):
        """Triggered after research. Returns structured PASS/FAIL verdict."""
        self.state.execution_log.append("[fact_check] start")

        agent = Agent(
            role=agents_config["fact_checker"]["role"],
            goal=agents_config["fact_checker"]["goal"],
            backstory=agents_config["fact_checker"]["backstory"],
            tools=[FactCheckTool()],
            llm=self._llm, verbose=True,
        )
        task = Task(
            description=(
                f"Fact-check this research:\n\n{self.state.research_output}\n\n"
                "Return JSON: claims_checked (int), verdicts (list of "
                "{claim, verdict, confidence}), overall_verdict (PASS or FAIL), "
                "recommendation (PROCEED_TO_WRITING or SEND_BACK_TO_RESEARCH)."
            ),
            expected_output=(
                "JSON with overall_verdict (exactly PASS or FAIL) and recommendation."
            ),
            agent=agent,
        )
        result = self._run_mini_crew(agent, task)
        self.state.fact_check_output = result

        verdict, rec = self._parse_verdict(result)
        self.state.fact_check_verdict = verdict
        self.state.fact_check_recommendation = rec

        self.state.execution_log.append(
            f"[fact_check] verdict={verdict} | rec={rec}"
        )

    # ─── Router: Branch on Verdict ───────────────────────────────────────

    @router(fact_check)
    def route_after_fact_check(self) -> str:
        """
        DETERMINISTIC router — no LLM call needed.
        Reads typed state fields directly.

        GUARDRAIL: if retries >= MAX_RESEARCH_RETRIES → escalate, don't loop.
        Without this check: FAIL → research → FAIL → research → ∞
        """
        if self.state.fact_check_verdict == "PASS":
            self.state.execution_log.append("[router] → write_article")
            return "write_article"

        # FAIL path
        if self.state.research_retry_count >= self.state.MAX_RESEARCH_RETRIES:
            self.state.execution_log.append(
                f"[router] → escalate_to_human "
                f"(retries exhausted {self.state.research_retry_count}/{self.state.MAX_RESEARCH_RETRIES})"
            )
            return "escalate_to_human"

        # Retry is within budget
        self.state.research_retry_count += 1
        self.state.execution_log.append(
            f"[router] → research (retry {self.state.research_retry_count})"
        )
        return "research"

    # ─── Step 3a: Write Article (PASS path) ─────────────────────────────

    @listen("write_article")
    def write_article(self):
        """Only reachable when fact-check PASSES."""
        self.state.execution_log.append("[write_article] start")

        agent = Agent(
            role=agents_config["writer"]["role"],
            goal=agents_config["writer"]["goal"],
            backstory=agents_config["writer"]["backstory"],
            tools=[], llm=self._llm, verbose=True,
        )
        task = Task(
            description=(
                f"Write a publication-ready article on: {self.state.topic}\n\n"
                f"Verified research:\n{self.state.research_output}\n\n"
                f"Fact-check report:\n{self.state.fact_check_output}\n\n"
                "Structure: Headline → Lead → Body (3-4 paragraphs) → Conclusion.\n"
                "Omit CONTRADICTED claims. Tag UNVERIFIED with [UNVERIFIED]."
            ),
            expected_output=(
                "Complete article in markdown, 400-600 words, "
                "with metadata footer: author, fact_check_status=PASS, topic."
            ),
            agent=agent,
        )
        self.state.final_article = self._run_mini_crew(agent, task)
        self.state.execution_log.append("[write_article] done")

    # ─── Step 3b: Escalate to Human (retry cap reached) ─────────────────

    @listen("escalate_to_human")
    def escalate_to_human(self):
        """
        GUARDRAIL terminal state.
        In production: page on-call, create Jira ticket, send Slack alert.
        """
        self.state.execution_log.append("[escalate_to_human] TRIGGERED")
        report = {
            "status": "ESCALATED",
            "reason": "Fact-check failed after maximum retries",
            "topic": self.state.topic,
            "retries_attempted": self.state.research_retry_count,
            "action_required": "Manual review and re-research required",
        }
        self.state.final_article = f"[ESCALATION]\n{json.dumps(report, indent=2)}"
        print("\n⚠️  ESCALATED: max retries reached — human review required")


print("✅ ResearchDepartmentFlow defined")
print("   Steps: research → fact_check → [write_article | research(retry) | escalate_to_human]")

### 3C — Run the Flow

In [ ]:
print("=" * 60)
print("  PHASE 3 — EVENT-DRIVEN FLOW EXECUTION")
print("=" * 60)

flow = ResearchDepartmentFlow(topic=TOPIC)
flow.kickoff()

print("\n" + "=" * 60)
print("EXECUTION LOG (audit trail from structured state):")
print("=" * 60)
for entry in flow.state.execution_log:
    print(f"  {entry}")

print(f"\nTool failure occurred  : {flow.state.tool_failure_occurred}")
print(f"Fact-check verdict     : {flow.state.fact_check_verdict}")
print(f"Research retries used  : {flow.state.research_retry_count} / {flow.state.MAX_RESEARCH_RETRIES}")

print("\n" + "=" * 60)
print("FINAL OUTPUT:")
print("=" * 60)
print(flow.state.final_article or "[No article — see escalation report above]")

### 3D — Inspect Full Pydantic State

In [ ]:
# Serialize entire state to JSON — useful for logging, replay, debugging
state_dict = flow.state.model_dump()

# Truncate long text fields for display
for key in ["research_output", "fact_check_output", "final_article"]:
    if state_dict.get(key):
        state_dict[key] = state_dict[key][:200] + "..."

print(json.dumps(state_dict, indent=2, default=str))

### Phase 3 — Questions & Answers

**Q1: What is the advantage of passing structured state (JSON/Pydantic) instead of raw text?**

> **(1) Schema enforcement** — Pydantic rejects malformed data immediately; raw text silently breaks downstream parsing.  
> **(2) Typed routing** — the `@router` reads `state.fact_check_verdict` (a `Literal["PASS","FAIL"]`) directly — no LLM call needed to interpret freeform text.  
> **(3) Debuggability** — `flow.state.model_dump()` gives a full JSON snapshot at any point, serializable for logging or replay.  
> **(4) IDE support** — autocomplete and type checking work on state fields, not regex patterns on strings.  
> **(5) Auditability** — `execution_log` in the state model provides a built-in step-by-step audit trail.

**Q2: What guardrail was implemented to prevent infinite loops between agents?**

> The `ResearchFlowState` carries two guardrail fields: `research_retry_count` (incremented each time the router sends the flow back to `research()` after a FAIL) and `MAX_RESEARCH_RETRIES` (hard cap, default = 2). In `route_after_fact_check()`, **before** routing back to `research`, the router checks:  
> ```python
> if self.state.research_retry_count >= self.state.MAX_RESEARCH_RETRIES:
>     return "escalate_to_human"   # terminal state — loop cannot continue
> ```
> The `escalate_to_human` step emits a structured report and **never** re-enters the research/fact-check cycle. Without this guard, a consistently low-quality research output would cause perpetual `FAIL → research → FAIL → ...` until the API budget was exhausted.

---
## Final Summary — All Phases


In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════╗
║           CREWAI HANDS-ON LAB — IMPLEMENTATION SUMMARY          ║
╠══════════════════════════════════════════════════════════════════╣
║ PHASE 1: Orchestration & Failure Handling                        ║
║  Sequential  → cheap, ordered, halts on unrecovered tool failure ║
║  Hierarchical→ resilient, Manager LLM reroutes failures, ~2x $$  ║
╠══════════════════════════════════════════════════════════════════╣
║ PHASE 2: YAML Config & Memory                                    ║
║  expected_output > backstory (last instruction wins)             ║
║  SQLite = long-term memory   ChromaDB = entity memory            ║
║  Both persist across runs; short-term resets each kickoff()      ║
╠══════════════════════════════════════════════════════════════════╣
║ PHASE 3: Event-Driven Flows                                      ║
║  Pydantic state → typed routing, no LLM needed for branching     ║
║  Guardrail: retry_count >= MAX_RETRIES → escalate_to_human       ║
║  Audit log: execution_log list in state for full traceability    ║
╚══════════════════════════════════════════════════════════════════╝
""")